#  Pivoting Data (`df.pivot()` vs. `df.pivot_table()`)

Pivoting is the process of reshaping data from a **long format** (many rows, few columns) to a **wide format** (fewer rows, more columns). It allows you to take unique values from a column and turn them into brand-new column headers, making the data much easier to read and analyze.

Pandas provides two primary tools for this:
1.  **`df.pivot()`**: Used for simple reshaping **without aggregation**. It simply reorganizes the table. If there are duplicate entries for the same index/column combination, it will crash and throw a `ValueError`.
2.  **`df.pivot_table()`**: A more advanced, spreadsheet-style pivot tool that **supports aggregation** (like calculating the mean, sum, or median of duplicates). This is the equivalent of Excel's pivot tables.

### Plain English Analogy
Imagine you have a messy **pile of receipts** (Long Format). Each receipt lists: `[Date, Category, Amount]`. You want to build a clean **weekly expense grid** (Wide Format) where:
*   The **rows** are the dates.
*   The **columns** are the categories (e.g., *Food*, *Rent*, *Transport*).
*   The **cells** show the amount spent.

If you only have one receipt per category per day, you can use **`pivot`**. But if you have multiple receipts for *Food* on the same day, you must use **`pivot_table`** to *sum* or *average* them together!

### Code Examples

First, let's create a DataFrame representing daily sales of different items:


In [1]:
import pandas as pd
import numpy as np

# Sample transactional sales data (Long Format)
data = {
    'Date': ['2026-08-01', '2026-08-01', '2026-08-02', '2026-08-02', '2026-08-03', '2026-08-03'],
    'Product': ['Laptops', 'Phones', 'Laptops', 'Phones', 'Laptops', 'Phones'],
    'Sales': [12000, 8000, 15000, 9500, 11000, 10000]
}

df = pd.DataFrame(data)
print("--- Original Long DataFrame ---")
print(df)

--- Original Long DataFrame ---
         Date  Product  Sales
0  2026-08-01  Laptops  12000
1  2026-08-01   Phones   8000
2  2026-08-02  Laptops  15000
3  2026-08-02   Phones   9500
4  2026-08-03  Laptops  11000
5  2026-08-03   Phones  10000


#### Simple Reshaping with `df.pivot()`
We want `Date` as the row index, unique `Product` names as new columns, and the `Sales` values in the cells:

In [2]:
# Pivoting long data to wide format
pivot_df = df.pivot(index='Date', columns='Product', values='Sales')
print("--- Reshaped Wide DataFrame (.pivot) ---")
print(pivot_df)

--- Reshaped Wide DataFrame (.pivot) ---
Product     Laptops  Phones
Date                       
2026-08-01    12000    8000
2026-08-02    15000    9500
2026-08-03    11000   10000


#### Aggregating Duplicates with `df.pivot_table()`
What if our source data has multiple entries for the same product on the same day? Let's see:

In [3]:
# Long data with duplicate entries (e.g., multiple stores reporting on the same day)
messy_data = {
    'Date': ['2026-08-01', '2026-08-01', '2026-08-01', '2026-08-02', '2026-08-02'],
    'Product': ['Laptops', 'Laptops', 'Phones', 'Laptops', 'Phones'],
    'Sales': [10000, 5000, 8000, 15000, 9500]  # Note two Laptop sales on 2026-08-01
}
df_messy = pd.DataFrame(messy_data)

# This would CRASH if we used df.pivot() because of the duplicates!
# Instead, we use pivot_table() and specify how to aggregate them (sum)
pivot_table_df = df_messy.pivot_table(index='Date', columns='Product', values='Sales', aggfunc='sum')
print("--- Aggregated Pivot Table (.pivot_table) ---")
print(pivot_table_df)

--- Aggregated Pivot Table (.pivot_table) ---
Product     Laptops  Phones
Date                       
2026-08-01    15000    8000
2026-08-02    15000    9500


*(Notice how the two Laptop entries of 10000 and 5000 on 2026-08-01 were automatically summed to 15000).*

### Common Pitfalls & Mistakes
*   **ValueError: Index contains duplicate entries**: This is the most common error when using `.pivot()`. If your dataset has duplicates for an index-column pair, you **must** use `.pivot_table()` instead and provide an aggregation function (like `aggfunc='sum'` or `aggfunc='mean'`).
*   **Forgetting to reset the index**: After pivoting, your index becomes the pivoted column (e.g., `Date`). If you want `Date` back as a regular column, you must follow up with `.reset_index()`.

#### Exercise 1 (Easy)
Given the following DataFrame:
```python
df_scores = pd.DataFrame({
    'Student': ['Alex', 'Alex', 'Sarah', 'Sarah'],
    'Subject': ['Math', 'Science', 'Math', 'Subject'],
    'Score': [95, 88, 90, 92]
})
```
Use `.pivot()` to reshape the data so that each student has one row, and the columns are `Math` and `Science` containing their scores.

In [4]:
df_scores = pd.DataFrame({
    'Student': ['Alex', 'Alex', 'Sarah', 'Sarah'],
    'Subject': ['Math', 'Science', 'Math', 'Subject'],
    'Score': [95, 88, 90, 92]
})

pivoted = df_scores.pivot(index='Student', columns='Subject', values='Score')
print(pivoted)

Subject  Math  Science  Subject
Student                        
Alex     95.0     88.0      NaN
Sarah    90.0      NaN     92.0


#### Exercise 2 (Medium)
Given a dataset of employee hours:
```python
df_hours = pd.DataFrame({
    'Date': ['Mon', 'Mon', 'Tue', 'Tue', 'Wed'],
    'Employee': ['John', 'John', 'John', 'Sarah', 'John'],
    'Hours': [4, 4, 8, 8, 6]  # John worked twice on Monday (4 hours each session)
})
```
Construct a pivot table calculating the *total* hours worked by each employee on each day. Fill any days they did not work with `0`. (Hint: Look up the `fill_value` parameter in the pivot table documentation).

In [5]:
df_hours = pd.DataFrame({
    'Date': ['Mon', 'Mon', 'Tue', 'Tue', 'Wed'],
    'Employee': ['John', 'John', 'John', 'Sarah', 'John'],
    'Hours': [4, 4, 8, 8, 6]  # John worked twice on Monday (4 hours each session)
})

# John worked two sessions on Monday. pivot_table sums them and fills Wed for Sarah with 0
pivot_hours = df_hours.pivot_table(index='Employee', columns='Date', values='Hours', aggfunc='sum', fill_value=0)
print(pivot_hours)

Date      Mon  Tue  Wed
Employee               
John        8    8    6
Sarah       0    8    0
